In [1]:
from pymongo import MongoClient
from collections import defaultdict
from datetime import datetime


In [2]:
client = MongoClient("mongodb://localhost:27017/")
db = client["Retail_Business"]

orders = db["Orders"]
seasonal_wh = db["seasonal_product_warehouse"]

In [3]:
raw_orders = list(orders.find())
len(raw_orders)


200000

In [4]:
seasonal = defaultdict(lambda: {
    "month": None,
    "product_name": None,
    "total_quantity": 0
})

for doc in raw_orders:
    month = doc["sale"]["date"][:7]
    product = doc["product"]["name"]

    key = (month, product)
    row = seasonal[key]

    row["month"] = month
    row["product_name"] = product
    row["total_quantity"] += doc["product"]["quantity"]


In [5]:
seasonal_wh.delete_many({})
seasonal_wh.insert_many(list(seasonal.values()))

seasonal_wh.count_documents({})


189365